## Compute EFPs, Store in CSVs

Using `measure='ee'` (appropriate for e+e- events), which uses energy fractions z_i = E_i / sum(E_j) and pairwise angles theta_ij computed from the 3-momenta directions.

Input to `batch_compute`: list of (N_particles, 4) arrays with columns (E, px, py, pz).

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import energyflow as ef
from energyflow.utils import gen_massless_phase_space
from energyflow import EFPSet
import os
from RegressFunctions import *
from RestrictedEFPs import efps_chunked
from PhysicsFunctions import *

os.makedirs('figures', exist_ok=True)

## Load Events

In [4]:
##### Load Pythia Events #####
def load_events(filepath):
    events, current = [], []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if line:
                current.append([float(x) for x in line.split()])
            elif current:
                events.append(np.array(current))
                current = []
    if current:
        events.append(np.array(current))
    return events

Zqq_events = load_events('data/Zqq_events.txt')
Hgg_events = load_events('data/Hgg_events.txt')


mults_Z = [len(e) for e in Zqq_events]
mults_H = [len(e) for e in Hgg_events]
print(f'Z->qqbar: {len(Zqq_events)} events, avg multiplicity {np.mean(mults_Z):.1f} +/- {np.std(mults_Z):.1f}')
print(f'H->gg:    {len(Hgg_events)} events, avg multiplicity {np.mean(mults_H):.1f} +/- {np.std(mults_H):.1f}')


#### Uniform Phase Space ####

NEvents=100_000
uniform_N3 = gen_massless_phase_space(NEvents,3,energy=1.0)
uniform_N5 = gen_massless_phase_space(NEvents,5,energy=1.0)
uniform_N7 = gen_massless_phase_space(NEvents,7,energy=1.0)
uniform_N10 = gen_massless_phase_space(NEvents,10,energy=1.0)

Z->qqbar: 100000 events, avg multiplicity 12.4 +/- 4.2
H->gg:    100000 events, avg multiplicity 19.9 +/- 5.2


## Compute EFPs, Save

In [7]:
n_jobs = len(os.sched_getaffinity(0))   # cores SLURM actually gave you
print(f'using {n_jobs} cores')

using 32 cores


In [9]:
efpset = ef.EFPSet(('d<=', 8),  measure='eeefm', beta=2, normed=True, coords='epxpypz')
print(f'EFP basis: {efpset.count()} EFPs')

EFPs_d8_N3 = efpset.batch_compute(uniform_N3, n_jobs=n_jobs);   print('uniform N=3 done')
EFPs_d8_N5 = efpset.batch_compute(uniform_N5, n_jobs=n_jobs);   print('uniform N=5 done')
EFPs_d8_N7 = efpset.batch_compute(uniform_N7, n_jobs=n_jobs);   print('uniform N=7 done')
EFPs_d8_N10 = efpset.batch_compute(uniform_N10, n_jobs=n_jobs);   print('uniform N=10 done')

EFPs_d8_Zqq = efpset.batch_compute(Zqq_events, n_jobs=n_jobs); print('Z->qqbar done')
EFPs_d8_Hgg = efpset.batch_compute(Hgg_events, n_jobs=n_jobs); print('H->gg done')


EFP basis: 3389 EFPs


: 

In [ ]:
#### Save as .npz ####
efp_sets = {
    'd8_uniform_N3':  EFPs_d8_N3,
    'd8_uniform_N5':  EFPs_d8_N5,
    'd8_uniform_N7':  EFPs_d8_N7,
    'd8_uniform_N10': EFPs_d8_N10,
    'd8_Zqq':         EFPs_d8_Zqq,
    'd8_Hgg':         EFPs_d8_Hgg,
}

np.savez_compressed(
    'EFPs_d8_beta2.npz',
    **{k: np.asarray(v) for k, v in efp_sets.items()},
    specs=efpset.specs,   # graph info per column (n, e, d, ...), handy for later filtering
)
print({k: np.asarray(v).shape for k, v in efp_sets.items()})

{'d4_uniform_N3': (100000, 36), 'd4_uniform_N5': (100000, 36), 'd4_uniform_N7': (100000, 36), 'd4_uniform_N10': (100000, 36), 'd4_Zqq': (100000, 36), 'd4_Hgg': (100000, 36)}
